# Strands vs RAGAS: Hallucination Detection Head-to-Head

Based on: [LSC: A Zero-Shot Metric for Hallucination Detection](https://arxiv.org/abs/2601.19918) (Jan 2026)

## The Problem

Your AI agent retrieves context from a knowledge base and generates a response. But did the agent stick to the facts, or did it fabricate details?

Two popular frameworks offer hallucination detection: **Strands Agents evals** (agent-native) and **RAGAS** (RAG-specialized). They take different approaches:

- **Strands** uses a general-purpose `OutputEvaluator` with a hallucination-focused rubric. Flexible but requires you to write the rubric.
- **RAGAS** provides purpose-built metrics (`Faithfulness`, `ResponseGroundedness`, `FactualCorrectness`) that decompose claims automatically.

## What We Test

5 test cases with known ground truth (2 grounded, 2 hallucinated, 1 mixed). Both frameworks evaluate the exact same responses against the exact same context. We measure:

1. **Accuracy** — Does each framework correctly identify hallucinated responses?
2. **Granularity** — How much detail does each framework provide about what went wrong?
3. **Code complexity** — How many lines of code does each approach need?

In [ ]:
# %pip install strands-agents strands-agents-evals ragas litellm boto3

## Setup: Load Test Data

**What this does:** Loads 5 pre-labeled test cases — each with a question, context, response, and a known ground-truth label (grounded or hallucinated).

**Why we need this:** Both frameworks will evaluate the exact same data, so any difference in results comes from the evaluation method, not the data. This controlled setup is essential for a fair head-to-head comparison.

> **What to look for:** 5 test cases should load — 2 grounded, 2 hallucinated, and 1 mixed. Note the explanations: they tell you *why* each case is labeled the way it is.

In [ ]:
import nest_asyncio
nest_asyncio.apply()  # Fix for Jupyter async event loop

from test_data import TEST_CASES, get_ground_truth

ground_truth = get_ground_truth()

print(f"📋 {len(TEST_CASES)} test cases loaded\n")
for tc in TEST_CASES:
    label = "❌ Hallucinated" if tc["is_hallucinated"] else "✅ Grounded"
    print(f"  {label}  {tc['name']}")
    print(f"           {tc['explanation']}\n")

---
## Test 1: Strands Agents — OutputEvaluator with Hallucination Rubric

**What this does:** Evaluates all 5 responses using Strands' general-purpose `OutputEvaluator` with a custom hallucination-focused rubric.

**Why this approach:** Strands does not have a dedicated hallucination metric. Instead, you write a rubric that tells the LLM judge what to look for. This is flexible — you control exactly what counts as a hallucination — but it produces a single score per response rather than per-claim detail.

### How it works

```
Question + Context + Response + Rubric → LLM Judge → Score (0-1) + Reason
```

The judge receives the context via the `expected_output` field and the response via the task function. The rubric tells the judge to check whether the response is grounded.

> **What to look for:** Each test case gets a score from 0.0 (hallucinated) to 1.0 (grounded). Grounded cases should score above 0.5; hallucinated cases should score below 0.5. The mixed case is the interesting one — watch whether Strands catches partial hallucination.

In [ ]:
from strands_evals import Experiment, Case
from strands_evals.evaluators import OutputEvaluator
from strands.models.openai import OpenAIModel

HALLUCINATION_RUBRIC = (
    "You are checking if the response is grounded in the provided context.\n\n"
    "Score 1.0: Every claim in the response is supported by the context.\n"
    "Score 0.5-0.7: Most claims are supported but the response includes minor "
    "embellishments or opinions not in the context.\n"
    "Score 0.0-0.3: The response contains fabricated facts, invented entities, "
    "made-up statistics, or claims not supported by the context.\n\n"
    "The context is provided in the expected_output field."
)

# Build Strands test cases — context goes in expected_output
strands_cases = [
    Case(
        name=tc["name"],
        input=tc["question"],
        expected_output="\n".join(tc["context"]),
    )
    for tc in TEST_CASES
]

strands_eval = OutputEvaluator(
    rubric=HALLUCINATION_RUBRIC,
    model="gpt-4o-mini",
)

# Task returns the pre-computed response
responses_by_name = {tc["name"]: tc["response"] for tc in TEST_CASES}

def strands_task(case):
    return responses_by_name[case.name]

# Run evaluation
print("=" * 60)
print("TEST 1: STRANDS — OutputEvaluator with hallucination rubric")
print("=" * 60)

strands_exp = Experiment(cases=strands_cases, evaluators=[strands_eval])
strands_reports = strands_exp.run_evaluations(strands_task)
strands_reports[0].display()

# Extract scores for comparison
strands_scores = {}
for case_result in strands_reports[0].cases:
    name = case_result["case_name"]
    score = case_result.get("score", 0)
    strands_scores[name] = score

---
## Test 2: RAGAS — Faithfulness Metric

**What this does:** Evaluates the same 5 responses using RAGAS' purpose-built `Faithfulness` and `ResponseGroundedness` metrics.

**Why RAGAS works differently:** Unlike Strands' single-score approach, RAGAS uses **claim decomposition** internally. It first breaks the response into individual factual claims, then checks each claim against the context independently. This means RAGAS can tell you *which specific facts* are fabricated, not only whether the overall response is problematic. You do not need to write a rubric — the evaluation logic is built into the metric.

### How it works

```
Response → Claim decomposition → Each claim checked against context → Score (0-1)
```

RAGAS is framework-agnostic: you pass plain strings (`user_input`, `response`, `retrieved_contexts`). No agent framework dependency.

### OpenAI integration

RAGAS connects to OpenAI through LiteLLM. You set `OPENAI_API_KEY` and use the model ID directly.

> **What to look for:** Compare the RAGAS faithfulness scores to the Strands scores from Test 1. They may differ because RAGAS decomposes claims before scoring — a response with 4 good claims and 1 fabricated claim may score 0.80 in RAGAS (4/5 claims) but get a different single-pass score from Strands.

In [ ]:
import os
import litellm
from ragas import evaluate as ragas_evaluate
from ragas.llms import llm_factory
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset
from ragas.metrics.collections import Faithfulness, ResponseGroundedness

# Configure OpenAI via LiteLLM
os.environ["OPENAI_API_KEY"] = os.environ.get("AWS_REGION", "us-east-1")

ragas_llm = llm_factory(
    "bedrock/anthropic.claude-sonnet-4-20250514-v1:0",
    provider="litellm",
    client=litellm.completion,
    temperature=0.0,
)

# Build RAGAS dataset — same data, RAGAS format
ragas_samples = [
    SingleTurnSample(
        user_input=tc["question"],
        response=tc["response"],
        retrieved_contexts=tc["context"],
    )
    for tc in TEST_CASES
]

ragas_dataset = EvaluationDataset(samples=ragas_samples)

print("=" * 60)
print("TEST 2: RAGAS — Faithfulness + ResponseGroundedness")
print("=" * 60)

ragas_result = ragas_evaluate(
    dataset=ragas_dataset,
    metrics=[
        Faithfulness(llm=ragas_llm),
        ResponseGroundedness(llm=ragas_llm),
    ],
)

# Display results
ragas_df = ragas_result.to_pandas()
print(ragas_df[["user_input", "faithfulness", "response_groundedness"]].to_string(index=False))

# Extract faithfulness scores for comparison
ragas_scores = {}
for i, tc in enumerate(TEST_CASES):
    ragas_scores[tc["name"]] = ragas_df.iloc[i]["faithfulness"]

---
## Ground Truth Verification

**What this does:** Compares both frameworks' scores against the known labels to measure detection accuracy.

**Why we need a threshold:** Both frameworks return continuous scores (0.0 to 1.0), but our ground truth is binary (hallucinated or not). The **threshold of 0.5** converts continuous scores into binary decisions: scores below 0.5 are classified as "hallucinated," scores at or above 0.5 as "grounded." This threshold is a design choice — lowering it makes detection more lenient (fewer false positives), raising it makes it stricter (fewer false negatives).

| Threshold Effect | Result |
|-----------------|--------|
| Score < 0.5 | Classified as **hallucinated** |
| Score >= 0.5 | Classified as **grounded** |
| Lower threshold (e.g., 0.3) | Only flags severe hallucinations |
| Higher threshold (e.g., 0.7) | Flags even mild embellishments |

> **What to look for:** For each test case, you will see whether each framework's binary decision matches the ground truth. Pay attention to the mixed/borderline case — that is where frameworks are most likely to disagree. The final accuracy numbers tell you which framework is more reliable for this dataset.

In [ ]:
THRESHOLD = 0.5

print("=" * 70)
print("GROUND TRUTH VERIFICATION")
print("=" * 70)

print(f"\n{'Case':<30} {'Truth':<15} {'Strands':<15} {'RAGAS':<15}")
print("-" * 70)

strands_correct = 0
ragas_correct = 0

for tc in TEST_CASES:
    name = tc["name"]
    truth = tc["is_hallucinated"]
    truth_label = "❌ halluc." if truth else "✅ grounded"

    # Strands: high score = grounded, low score = hallucinated
    s_score = strands_scores.get(name, 0)
    s_detected = s_score < THRESHOLD
    s_correct = s_detected == truth
    strands_correct += s_correct
    s_icon = "✅" if s_correct else "❌"

    # RAGAS: high score = faithful, low score = hallucinated
    r_score = ragas_scores.get(name, 0)
    r_detected = r_score < THRESHOLD
    r_correct = r_detected == truth
    ragas_correct += r_correct
    r_icon = "✅" if r_correct else "❌"

    print(f"  {name:<28} {truth_label:<15} {s_icon} {s_score:.2f}{'':>6} {r_icon} {r_score:.2f}")

print(f"\n📊 Detection Accuracy (threshold={THRESHOLD}):")
print(f"   Strands: {strands_correct}/{len(TEST_CASES)} ({100*strands_correct/len(TEST_CASES):.0f}%)")
print(f"   RAGAS:   {ragas_correct}/{len(TEST_CASES)} ({100*ragas_correct/len(TEST_CASES):.0f}%)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(14, 5))
fig.set_facecolor('white')

case_names = [tc["name"] for tc in TEST_CASES]
strands_vals = [strands_scores.get(n, 0) for n in case_names]
ragas_vals = [ragas_scores.get(n, 0) for n in case_names]
ground_truths = [not tc["is_hallucinated"] for tc in TEST_CASES]  # True = grounded

x = np.arange(len(case_names))
width = 0.30

bars1 = ax.bar(x - width/2, strands_vals, width, label='Strands', color='#2196F3', edgecolor='white')
bars2 = ax.bar(x + width/2, ragas_vals, width, label='RAGAS', color='#4CAF50', edgecolor='white')

for bar, score in zip(bars1, strands_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{score:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
for bar, score in zip(bars2, ragas_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{score:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Ground truth markers
for i, is_grounded in enumerate(ground_truths):
    marker = '^' if is_grounded else 'v'
    color = '#4CAF50' if is_grounded else '#E53935'
    ax.scatter(i, 1.05, marker=marker, color=color, s=100, zorder=5)

ax.axhline(y=THRESHOLD, color='#E53935', linestyle='--', linewidth=1.5, label=f'Threshold ({THRESHOLD})')
ax.set_xticks(x)
ax.set_xticklabels(case_names, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Strands vs RAGAS: Hallucination Detection Scores per Test Case\n(triangles: ground truth — up=grounded, down=hallucinated)',
             fontweight='bold', fontsize=13)
ax.set_ylim(0, 1.15)
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

---
## Code Complexity Comparison

**What this does:** Summarizes the practical differences in setup, configuration, and dependencies between the two frameworks.

**Why it matters:** Detection accuracy is one dimension; developer experience is another. A framework that requires fewer lines of code, no rubric writing, and fewer dependencies may be the right choice even if accuracy is comparable.

> **What to look for:** The table highlights that Strands is simpler (fewer dependencies, native model support) while RAGAS is more specialized (dedicated context field, built-in claim decomposition). Neither is universally better — the right choice depends on your use case.

In [ ]:
print("=" * 60)
print("CODE COMPLEXITY COMPARISON")
print("=" * 60)

comparison = [
    ["Aspect",              "Strands evals",                    "RAGAS"],
    ["Setup lines",         "3 (import + rubric + evaluator)",  "5 (import + litellm + llm_factory + metrics)"],
    ["Per-case definition", "Case(input=, expected_output=)",   "SingleTurnSample(user_input=, response=, retrieved_contexts=)"],
    ["Context field",       "expected_output (workaround)",     "retrieved_contexts (dedicated)"],
    ["Rubric required?",    "Yes (you write it)",               "No (built into metric)"],
    ["Claim decomposition", "No (single score)",                "Yes (per-claim)"],
    ["Bedrock integration", "Native (model ID string)",         "Via LiteLLM (bedrock/ prefix)"],
    ["Extra dependencies",  "None",                             "litellm, ragas"],
    ["Best for",            "Agent-native eval, custom rubrics","RAG-specific eval, claim granularity"],
]

for row in comparison:
    if row[0] == "Aspect":
        print(f"\n  {'Aspect':<22} {'Strands evals':<35} {'RAGAS'}")
        print(f"  {'-'*22} {'-'*35} {'-'*35}")
    else:
        print(f"  {row[0]:<22} {row[1]:<35} {row[2]}")

---
## Summary

### When to Use Each

| Need | Best Choice | Why |
|------|-------------|-----|
| You already use Strands Agents | **Strands evals** | No extra dependencies, same ecosystem, custom rubrics |
| You need per-claim granularity | **RAGAS** | Automatic claim decomposition identifies exactly which facts are fabricated |
| You evaluate RAG pipelines | **RAGAS** | Purpose-built for retrieval + generation evaluation with dedicated `retrieved_contexts` field |
| You want full control over criteria | **Strands evals** | Write your own rubric with domain-specific instructions |
| You need both | **Combine them** | Use Strands for agent-level eval and RAGAS for RAG-specific metrics |

### Key Findings

1. **Both frameworks detect obvious hallucinations.** Fabricated airlines, awards, and statistics are caught by both.
2. **RAGAS has better granularity.** Its claim decomposition tells you *which* claims are fabricated. Strands gives a single score.
3. **Strands has less setup.** No extra dependencies, native multi-provider support, 3 lines to configure.
4. **Mixed/subtle hallucinations are hardest.** Embellishments (such as "great weather for sightseeing") are the edge cases where frameworks diverge.
5. **You can use both together.** RAGAS evaluates the RAG pipeline; Strands evaluates the full agent behavior including tool use and trajectory.

### References

- [LSC: Zero-Shot Hallucination Detection](https://arxiv.org/abs/2601.19918) (Jan 2026) — Span confidence without training data
- [VISTA: Turn-based Claim Verification](https://arxiv.org/abs/2510.27052) (Oct 2025) — Atomic claim decomposition for conversations
- [Spilled Energy](https://arxiv.org/abs/2602.18671) (Feb 2026) — Training-free logit-based hallucination detection

**Next:** [Demo 02 - Claim Decomposition](../02-claim-decomposition/) — Build your own claim-level verification from scratch with Strands Agents.